In [3]:
import pandas as pd
df=pd.read_csv("rainfall.csv")
df.head()

,datetime,temp,dew,humidity,sealevelpressure,winddir,solarradiation,windspeed,precipprob,preciptype
0,01-01-2016,28.4,11.9,37.8,1016.4,147.5,216.1,16.6,0,0
1,02-01-2016,26.8,13.0,44.8,1017.2,110.3,215.7,16.6,0,0
2,03-01-2016,25.5,14.6,52.8,1015.7,145.3,221.1,18.4,0,0
3,04-01-2016,26.4,13.1,46.6,1015.3,126.9,216.2,16.6,0,0
4,05-01-2016,27.1,13.5,44.9,1014.4,125.5,208.1,16.6,0,0


In [4]:
df.drop(columns=['datetime', 'winddir', 'solarradiation', 'precipprob'], inplace=True)


In [5]:
df.head()

,temp,dew,humidity,sealevelpressure,windspeed,preciptype
0,28.4,11.9,37.8,1016.4,16.6,0
1,26.8,13.0,44.8,1017.2,16.6,0
2,25.5,14.6,52.8,1015.7,18.4,0
3,26.4,13.1,46.6,1015.3,16.6,0
4,27.1,13.5,44.9,1014.4,16.6,0


In [4]:
#converting the sealvelpressure from hpa to pascal
df['sealevelpressure'] = df['sealevelpressure'] * 100
#converting the windspeed from km/h to m/s
df['windspeed'] = df['windspeed'] * (1000/3600)


In [6]:
#analyzing the class distribution of the target variable
df['preciptype'].value_counts()

preciptype
0    968
1    813
Name: count, dtype: int64

In [7]:
df.describe()

,temp,dew,humidity,sealevelpressure,windspeed,preciptype
count,1781.000000,1781.000000,1781.000000,1781.000000,1781.000000,1781.000000
mean,28.342560,21.644975,69.565974,1008.878046,22.309882,0.456485
std,1.960027,4.532065,14.533809,3.706157,6.360669,0.498243
min,20.200000,5.000000,28.600000,994.100000,9.400000,0.000000
25%,27.200000,18.100000,58.000000,1006.200000,18.400000,0.000000
50%,28.500000,23.800000,71.600000,1009.100000,22.300000,0.000000
75%,29.700000,25.300000,81.200000,1011.800000,24.600000,1.000000
max,32.800000,27.500000,98.200000,1017.400000,128.100000,1.000000


In [9]:
df.isnull().sum()

temp                0
dew                 0
humidity            0
sealevelpressure    0
windspeed           0
preciptype          0
dtype: int64

In [8]:
X=df.drop(columns=['preciptype'])
y=df['preciptype']


In [9]:
#creating a ml pipeline for preprocessing and training the model
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,AdaBoostClassifier,StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score,balanced_accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, fbeta_score
# Creating a column transformer for preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric_features = ['temp', 'dew', 'humidity', 'windspeed', 'sealevelpressure']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())])
model_pipeline_xgb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features)
    ])),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
])
#applying the model pipeline to the training data
model_pipeline_xgb.fit(X_train, y_train)
#applying the grid search to find the best hyperparameters for the model
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.1, 0.2]
}
f2_scorer = make_scorer(fbeta_score, beta=2, average='weighted')
grid_search = GridSearchCV(model_pipeline_xgb, param_grid, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search.fit(X_train, y_train)



c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [17:50:25] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\LOQ\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\core.py:158: UserWarning: [17:50:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [10]:
#evaluating the model on the test set
y_pred = grid_search.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.97      0.91       184
           1       0.96      0.82      0.88       173

    accuracy                           0.90       357
   macro avg       0.91      0.89      0.90       357
weighted avg       0.90      0.90      0.90       357

Confusion Matrix:
 [[178   6]
 [ 31 142]]
Balanced Accuracy Score: 0.8941002764513697


In [11]:
# training by AdaBoostClassifier
model_pipeline_ada = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated'))
])
model_pipeline_ada.fit(X_train, y_train)
#finding the best hyperparameters for AdaBoostClassifier using grid search
param_grid_ada = {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1.0]
}
grid_search_ada = GridSearchCV(model_pipeline_ada, param_grid_ada, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_ada.fit(X_train, y_train)



,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [12]:
#evaluating the AdaBoostClassifier model on the test set
y_pred_ada = grid_search_ada.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_ada))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_ada))

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.96      0.91       184
           1       0.95      0.85      0.90       173

    accuracy                           0.91       357
   macro avg       0.91      0.91      0.91       357
weighted avg       0.91      0.91      0.91       357

Confusion Matrix:
 [[177   7]
 [ 26 147]]
Balanced Accuracy Score: 0.9058337521990449


In [13]:
#training with GradientBoostingClassifier
model_pipeline_gb = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42))
])
model_pipeline_gb.fit(X_train, y_train)
param_grid_gb = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.01, 0.1, 0.2],
    'classifier__max_depth': [3, 5, 7],
    
}
grid_search_gb = GridSearchCV(model_pipeline_gb, param_grid_gb, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_gb.fit(X_train, y_train)


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__learning_rate': [0.01, 0.1, ...], 'classifier__max_depth': [3, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [14]:
#evaluating the gradient boosting model on the test set
y_pred_gb = grid_search_gb.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_gb))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_gb))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_gb))


Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.95      0.92       184
           1       0.94      0.87      0.90       173

    accuracy                           0.91       357
   macro avg       0.91      0.91      0.91       357
weighted avg       0.91      0.91      0.91       357

Confusion Matrix:
 [[175   9]
 [ 23 150]]
Balanced Accuracy Score: 0.9090694898215632


In [15]:
#training with random forest classifier
model_pipeline_rf = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[('num', numeric_transformer, numeric_features)])),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))
])
model_pipeline_rf.fit(X_train, y_train)
param_grid_rf = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [3, 5, 7],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
}
grid_search_rf = GridSearchCV(model_pipeline_rf, param_grid_rf, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], 'classifier__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...)]"


In [16]:
#evalauting the random forest model on the test set
y_pred_rf = grid_search_rf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_rf))


Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.95      0.92       184
           1       0.94      0.87      0.90       173

    accuracy                           0.91       357
   macro avg       0.91      0.91      0.91       357
weighted avg       0.91      0.91      0.91       357

Confusion Matrix:
 [[174  10]
 [ 22 151]]
Balanced Accuracy Score: 0.90924227192762


In [17]:
#now training by StackingClassifier
base_estimators = [
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42,algorithm='deprecated')),
    ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss',learning_rate=0.01, n_estimators=50, max_depth=5))
]
meta_estimator = RandomForestClassifier(max_depth=7, n_estimators=50, min_samples_split=5, min_samples_leaf=2,random_state=42)
stacking_model = StackingClassifier(estimators=base_estimators, final_estimator=meta_estimator, cv=5, n_jobs=-1)
stacking_model.fit(X_train, y_train)
param_grid_stacking = {
    'final_estimator__n_estimators': [50, 100, 200],
    'final_estimator__max_depth': [3, 5, 7],
}
grid_search_stacking = GridSearchCV(stacking_model, param_grid_stacking, cv=5, scoring=f2_scorer, n_jobs=-1)
grid_search_stacking.fit(X_train, y_train)

,estimator,StackingClass... n_jobs=-1)
,param_grid,"{'final_estimator__max_depth': [3, 5, ...], 'final_estimator__n_estimators': [50, 100, ...]}"
,scoring,make_scorer(f...rage=weighted)
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,estimator,None


In [18]:
#evaluating the stacking model on the test set
y_pred_stacking = grid_search_stacking.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_stacking))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stacking))
print("Balanced Accuracy Score:", balanced_accuracy_score(y_test, y_pred_stacking))


Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.96      0.91       184
           1       0.95      0.84      0.89       173

    accuracy                           0.90       357
   macro avg       0.91      0.90      0.90       357
weighted avg       0.91      0.90      0.90       357

Confusion Matrix:
 [[176   8]
 [ 27 146]]
Balanced Accuracy Score: 0.9002261874842925


In [19]:

from sklearn.metrics import recall_score

model_results = {
    "XGBoost": grid_search,
    "AdaBoost": grid_search_ada,
    "GradientBoosting": grid_search_gb,
    "RandomForest": grid_search_rf,
    "StackingClassifier": grid_search_stacking,
}

print("Recall (class 1 = rain) comparison across all trained models:")
recalls = {}
for name, fitted_search in model_results.items():
    preds = fitted_search.predict(X_test)
    rec = recall_score(y_test, preds, pos_label=1)
    recalls[name] = rec
    print(f"  {name:20s} recall = {rec:.4f}")

best_model_name = max(recalls, key=recalls.get)
best_model = model_results[best_model_name]
print(f"\n>>> BEST MODEL BY RECALL: {best_model_name} ({recalls[best_model_name]:.4f})")
print("NOTE: if StackingClassifier does NOT win here, that's a real and reportable")
print("      result -- stacking doesn't always beat its own base models.")

Recall (class 1 = rain) comparison across all trained models:
  XGBoost              recall = 0.8208
  AdaBoost             recall = 0.8497
  GradientBoosting     recall = 0.8671
  RandomForest         recall = 0.8728
  StackingClassifier   recall = 0.8439

>>> BEST MODEL BY RECALL: RandomForest (0.8728)
NOTE: if StackingClassifier does NOT win here, that's a real and reportable
      result -- stacking doesn't always beat its own base models.


In [20]:

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve
import numpy as np

# Get out-of-fold probabilities on the TRAINING set only
oof_probs = cross_val_predict(
    best_model.best_estimator_, X_train, y_train,
    cv=5, method="predict_proba", n_jobs=-1
)[:, 1]  # probability of class 1 (rain)

precisions, recalls_curve, thresholds = precision_recall_curve(y_train, oof_probs, pos_label=1)

TARGET_RECALL = 0.9  # catch at least 90% of real heavy-rain events
valid_idx = np.where(recalls_curve[:-1] >= TARGET_RECALL)[0]

if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precisions[valid_idx])]
    best_threshold = thresholds[best_idx]
    print(f"Chosen threshold (from train-set OOF probabilities): {best_threshold:.3f}")
    print(f"Expected recall={recalls_curve[best_idx]:.3f}, precision={precisions[best_idx]:.3f}")
else:
    print(f"No threshold reaches {TARGET_RECALL} recall on OOF predictions.")
    print("Consider lowering TARGET_RECALL, e.g. to 0.85.")
    best_threshold = 0.5

Chosen threshold (from train-set OOF probabilities): 0.253
Expected recall=0.900, precision=0.766


In [21]:

from sklearn.metrics import classification_report, confusion_matrix, fbeta_score

test_probs = best_model.predict_proba(X_test)[:, 1]
test_preds_thresholded = (test_probs >= best_threshold).astype(int)

print(f"\n=== FINAL: {best_model_name} @ threshold={best_threshold:.3f} on held-out TEST set ===")
print(classification_report(y_test, test_preds_thresholded, target_names=["no_rain", "rain"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_preds_thresholded))
print(f"F2-score: {fbeta_score(y_test, test_preds_thresholded, beta=2, pos_label=1):.4f}")

print(f"\n>>> threshold = {best_threshold:.3f} with model = {best_model_name}")
print(">>> in ESP32 HAZARD_MODELS config for rainfall prediction.")


=== FINAL: RandomForest @ threshold=0.253 on held-out TEST set ===
              precision    recall  f1-score   support

     no_rain       0.93      0.83      0.88       184
        rain       0.84      0.93      0.88       173

    accuracy                           0.88       357
   macro avg       0.88      0.88      0.88       357
weighted avg       0.88      0.88      0.88       357

Confusion matrix:
[[153  31]
 [ 12 161]]
F2-score: 0.9106

>>> threshold = 0.253 with model = RandomForest
>>> in ESP32 HAZARD_MODELS config for rainfall prediction.


In [22]:
#now converting this random forest model to tflite by distillation 
teacher_model = best_model.best_estimator_
teacher_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [24]:
soft_labels = teacher_model.predict_proba(X_train)


In [25]:
import tensorflow as tf
from tensorflow.keras import layers, models
student_model = models.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(soft_labels.shape[1], activation='softmax')
])
student_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
student_model.fit(X_train, soft_labels, epochs=50, batch_size=32, validation_split=0.2)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4820 - loss: 15.5981 - val_accuracy: 0.4000 - val_loss: 2.8944
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3960 - loss: 1.6163 - val_accuracy: 0.5719 - val_loss: 0.9821
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4688 - loss: 0.8686 - val_accuracy: 0.4772 - val_loss: 0.6982
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5593 - loss: 0.7569 - val_accuracy: 0.6561 - val_loss: 0.5746
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6980 - loss: 0.5846 - val_accuracy: 0.7789 - val_loss: 0.5057
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7761 - loss: 0.5086 - val_accuracy: 0.8070 - val_loss: 0.4669
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7963 - loss: 0.4790 - val_accuracy: 0.7754 - val_loss: 0.4592
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7972 - loss: 0.4781 - val_accuracy: 0.7544 - val_loss

In [26]:
convertor=tf.lite.TFLiteConverter.from_keras_model(student_model)
convertor.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = convertor.convert()

with open("rainfall_predict_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpx4ttxh8t\assets


INFO:tensorflow:Assets written to: C:\Users\LOQ\AppData\Local\Temp\tmpx4ttxh8t\assets


Saved artifact at 'C:\Users\LOQ\AppData\Local\Temp\tmpx4ttxh8t'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 5), dtype=tf.float32, name='keras_tensor_1')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  1451899102912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1451899095696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1451899107136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1451899099392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1451899101504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1451899174976: TensorSpec(shape=(), dtype=tf.resource, name=None)
